# M5 Raw Data Exploration

Dataset Description

In the challenge, you are predicting item sales at stores in various locations for two 28-day time periods. Information about the data is found in the M5 Participants Guide.
Files

    calendar.csv - Contains information about the dates on which the products are sold.
    sales_train_validation.csv - Contains the historical daily unit sales data per product and store [d_1 - d_1913]
    sample_submission.csv - The correct format for submissions. Reference the Evaluation tab for more info.
    sell_prices.csv - Contains information about the price of the products sold per store and date.
    sales_train_evaluation.csv - Includes sales [d_1 - d_1941] (labels used for the Public leaderboard)

## 1. Libraries


In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "m5" / "raw"

calendar_path = RAW_DATA_DIR / "calendar.csv"
sales_path = RAW_DATA_DIR / "sales_train_validation.csv"
prices_path = RAW_DATA_DIR / "sell_prices.csv"

RAW_DATA_DIR

WindowsPath('c:/Users/Francisco/Documents/Documentos-secundario/Visual-Studio-Code-Repository/CT5108_Capstone/Unified_forecasting/data/m5/raw')

## 2. File Overview

Checks size of the raw files

In [ ]:
raw_files = []

for file_path in sorted(RAW_DATA_DIR.glob("*.csv")):
    raw_files.append({
        "file_name": file_path.name,
        "size_mb": round(file_path.stat().st_size / (1024 * 1024), 2),
    })

files_df = pd.DataFrame(raw_files)
files_df

## 3. Load Raw Data

Calendar, sales and price files

In [ ]:
calendar = pd.read_csv(calendar_path)
sales = pd.read_csv(sales_path) #, nrows=1000)
prices = pd.read_csv(prices_path) #, nrows=100000)

print("calendar shape:", calendar.shape)
print("sales sample shape:", sales.shape)
print("prices sample shape:", prices.shape)

## 4. Calendar Data exploration

The calendar table links M5 day numbers like **d_1** to real dates and event information.

In [ ]:
calendar.shape

In [ ]:
calendar.columns

In [ ]:
calendar.head(10)

In [ ]:
calendar.tail()

In [ ]:
calendar.info()

In [ ]:
calendar.dtypes

In [ ]:
calendar.isna().sum()

In [ ]:
calendar.describe(include="all")

In [ ]:
calendar.nunique()

In [ ]:
calendar["weekday"].unique()

In [ ]:
calendar["event_type_1"].value_counts(dropna=False)

In [ ]:
calendar.shape[0]/7

In [ ]:
calendar["weekday"].value_counts().plot(kind="bar", title="Weekday counts")
plt.xlabel("Weekday")
plt.ylabel("Number of rows")
plt.tight_layout()
plt.show()

The calendar data has one row per M5 day. It includes normal date fields, event columns, and SNAP indicator columns for different states.

## 5. Sales Data exploration

The sales table is wide 
Product and store details are in the first columns, then daily sales are stored in columns d_1, d_2, ..., d_n

Each row is a unique item_id-store_id

In [ ]:
sales.shape

In [ ]:
sales.columns

In [ ]:
sales.head(10)

In [ ]:
sales.info()

In [ ]:
sales.dtypes.head(15)

In [ ]:
sales.isna().sum().head(15)

In [ ]:
sales.describe()

In [ ]:
sales.nunique().head(15)

In [ ]:
sales["cat_id"].unique()

In [ ]:
sales["store_id"].unique()

In [ ]:
sales["cat_id"].value_counts()

In [ ]:
sales["store_id"].value_counts()

In [ ]:
sales["cat_id"].value_counts().plot(kind="bar", title="Items by Category")
plt.xlabel("Category")
plt.ylabel("Number of rows")
plt.tight_layout()
plt.show()

In [ ]:
sales["store_id"].value_counts().plot(kind="bar", title="Items by Store")
plt.xlabel("Store")
plt.ylabel("Number of rows")
plt.tight_layout()
plt.show()

In [ ]:
day_columns = [column for column in sales.columns if column.startswith("d_")]
first_n_days = day_columns[:100]

# Find an item with sales in the first 100 days
sales["first_100_total"] = sales[first_n_days].sum(axis=1)
item_index = sales["first_100_total"].idxmax()

one_item_sales = sales.loc[item_index, first_n_days]
one_item_sales.index = range(1, len(one_item_sales) + 1)

one_item_sales.plot(kind="line", title="One Item Sales first 100 Days")
plt.xlabel("Day number")
plt.ylabel("Units sold")
plt.tight_layout()
plt.show()

Sales must be transformed from wide to large format

## 6. Sell Prices Data Sample

The prices table track the selling price for each item, store, and Walmart week.

In [ ]:
prices.shape

In [ ]:
prices.columns

In [ ]:
prices.head(10)

In [ ]:
prices.info()

In [ ]:
prices.dtypes

In [ ]:
prices.isna().sum()

In [ ]:
prices.nunique()

In [ ]:
prices["store_id"].unique()

In [ ]:
prices["store_id"].value_counts()

In [ ]:
prices["sell_price"].plot(kind="hist", bins=100, title="Distribution of Prices")
plt.xlabel("Sell price")
plt.ylabel("Number of rows")
plt.tight_layout()
plt.show()

In [ ]:
prices.groupby("store_id")["sell_price"].mean().sort_values().plot(kind="bar", title="Average Price by Store")
plt.xlabel("Store")
plt.ylabel("Average sell price")
plt.tight_layout()
plt.show()

The sell price data is useful because prices can change over time. This table can be joined to sales data using `store_id`, `item_id`, and `wm_yr_wk`.

### Price over time example

In [ ]:
# Pick one item and store with fluctuating prices
price_changes = (
    prices
    .sort_values(["store_id", "item_id", "wm_yr_wk"])
    .groupby(["store_id", "item_id"])["sell_price"]
    .nunique()
    .sort_values(ascending=False)
)

price_changes.head()

In [ ]:
one_store, one_item = price_changes.index[0]

price_history = prices[
    (prices["item_id"] == one_item) &
    (prices["store_id"] == one_store)
].copy()

price_history = price_history.sort_values("wm_yr_wk")

price_history.plot(
    x="wm_yr_wk",
    y="sell_price",
    kind="line",
    marker="o",
    markersize=3,
    title=f"Price Over Time for: {one_item} in {one_store}"
)

plt.xlabel("Walmart year-week")
plt.ylabel("Sell price")
plt.tight_layout()
plt.show()


## Summary

- calendar.csv has dates properties and events for each `d_` day code.
- sales_train_validation.csv contains item/store identifiers and daily sales in wide format.
- sell_prices.csv has weekly prices by item and store.

Actions
- The sales table is wide, need to be melt to large format.
- The files can be joined using day/week, item, and store columns.